In [25]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
import random

import os
import pickle
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch
import copy

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, VARasNN
import functions

# Data Preparation

Do following for different risk aversions beta:

for date in test_dates:

1. prepare train and test data (date)

2. prepare scenario matrix out of train data

3. specify constants for cvxpy (dimensions of scenario matrix, etc.)

4. precompute oracle solution w*(c)

5. Train VAR on train with custom loss function (combinations of MSE and DFL loss)

    Do so with every combination
    Save relevant metrics
    Estimate returns c_hat and w*(c_hat)

-> Agreggate metrics ofer the whole test period (backtesting period)

In [23]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data.csv")

In [24]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,A.N,AAPL.OQ,ABT.N,ACGL.OQ,ADBE.OQ,ADI.OQ,ADM.N,ADP.OQ,ADSK.OQ,AEE.N,...,WMB.N,WMT.N,WST.N,WY.N,XEL.OQ,XOM.N,XRAY.OQ,YUM.N,ZBRA.OQ,ZION.OQ
date,,,,,,,,,,,,,,,,,,,,,
2000-01-31,-0.143897,0.009119,-0.097065,0.158416,-0.181227,0.005376,-0.035897,-0.119490,-0.092685,-5.725191e-03,...,0.267894,-0.207957,0.005608,-0.195522,-0.012821,0.036462,0.047619,-0.258900,0.011752,0.003960
2000-02-29,0.566572,0.104819,0.003831,0.042735,0.852440,0.679144,-0.139973,-0.081686,0.462168,-7.869482e-02,...,0.079032,-0.110731,-0.048485,-0.105664,-0.087662,-0.092836,0.035354,-0.069869,0.124604,-0.102537
2000-03-31,0.003014,0.184842,0.074427,0.073770,0.091363,0.026274,0.031056,0.109845,0.018182,5.474215e-02,...,0.054239,0.141254,-0.140127,0.110840,0.131673,0.033195,0.109834,0.166667,-0.248826,-0.215548
2000-04-28,-0.147837,-0.086516,0.097600,-0.060115,0.086468,-0.046548,-0.042169,0.115285,-0.155477,1.858586e-01,...,-0.150782,-0.002252,-0.027729,-0.062500,0.116715,-0.001606,0.024229,0.098592,0.140000,-0.003003
2000-05-31,-0.169252,-0.322922,0.058537,-0.025381,-0.069251,0.002441,0.207154,0.020906,-0.030945,-7.407630e-12,...,0.113903,0.040632,-0.038363,-0.064345,0.014327,0.078095,0.055914,-0.141026,-0.157895,0.131392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.076402,-0.088678,-0.058795,0.037080,-0.088390,-0.032262,-0.048928,-0.050382,-0.067721,-4.845823e-02,...,-0.011417,-0.016481,-0.077882,-0.063817,0.010451,0.057469,-0.075219,-0.034318,-0.139922,-0.017183
2023-10-31,-0.073692,-0.002570,-0.018228,0.087442,0.043460,-0.101434,-0.051047,-0.092942,-0.044850,1.175999e-02,...,0.021075,0.021760,-0.151702,-0.064253,0.035827,-0.099762,-0.109778,-0.032656,-0.114573,-0.115792
2023-11-30,0.236335,0.113747,0.103014,-0.034495,0.148386,0.165576,0.036457,0.053616,0.105247,2.483159e-02,...,0.069477,-0.047243,0.102681,0.099338,0.026489,-0.020540,0.044064,0.043727,0.131548,0.168970


In [11]:
X, Y = functions.create_time_series_data_with_lags(return_matrix, max_lag=3)        # HYPERPARAM: lags of VAR model

print(X.shape)
print(Y.shape)

(286, 45)
(286, 15)


## Try looping

In [12]:
beta_levels = [0.07, 0.08, 0.09, 0.10] # feasible CVaR levels
beta_levels = [0.09]

n_test_months = 12
n_test_months = 4

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]
gamma_levels = [0.2]

n_epochs = 10            # HYPERPARAM: number of epochs

In [18]:
test_months = [2]

In [19]:
val_length = 6              # HYPERPARAM: validation length

Problem: scenario loss marix is constant for all entries in X_train and in training it contains scenarios which are in the future.

Is it really a Problem?

I think its a feature: we trade of (very limited) data leakage in training for a richer scenario matrix which is actually the best possible for estimating the true risk at test date!


In [20]:
results = []

for beta in beta_levels:
    
    # for test_index in range(1,n_test_months+1):
    for test_index in tqdm(test_months, desc="test months"):

        print(f"\nBeta = {beta}; Test Index = {test_index}\n")
        print("Preparing Data...")

        # data preparation
        X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
        train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix, test_index + val_length, num_scenarios=1000, random_seed=42)
        S, N = train_scenario_loss_matrix.shape # S scenarios × N assets

        # solver
        train_solver = CVaRSolver(
            loss_matrix=train_scenario_loss_matrix,
            N=N,
            S=S,
            alpha=0.95, # CVaR confidence level
            beta=beta
        )

        # Precompute oracle solutions
        oracle_solutions = [
            train_solver.solve(c = - mu).copy()
            for mu in tqdm(Y_train, desc="Computing oracle solutions")
        ]
        oracle_tensor = torch.tensor(
            np.array(oracle_solutions),
            dtype=torch.float32
        )

        # Precompute oracle solutions for validation set
        oracle_val_solutions = [
            train_solver.solve(c=-mu).copy()
            for mu in tqdm(Y_val, desc="Computing validation oracle solutions")
        ]
        oracle_val_tensor = torch.tensor(
            np.array(oracle_val_solutions),
            dtype=torch.float32
        )


        # Prepare dataset for pytorch training
        x_scaler = StandardScaler()
        X_train = x_scaler.fit_transform(X_train)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
        train_dataset = TensorDataset(
            X_train_tensor,
            Y_train_tensor,
            oracle_tensor
        )
        batch_size = 16                                                          # HYPERPARAM: Batch size
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,        # TensorDataset shuffles all tensors together
            drop_last=False
        )

        # scale validation data using the training scaler
        X_val_scaled = x_scaler.transform(X_val)
        X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
        Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

        # specify model for scale computation
        model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
        criterion = nn.MSELoss()
        # compute scale factor for the two losses to make them comparable
        spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
        print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

        print("Train Models for different loss combinations:")
        
        for gamma in gamma_levels:
            
            print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

            # specify model and optimizer
            torch.manual_seed(42)
            model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=1e-3                                                             # HYPERPARAM: Learning rate
            )
            criterion = nn.MSELoss()

            # Train model and retrieve training history
            history = functions.train_model(model,
                                            n_epochs,
                                            train_loader,
                                            optimizer,
                                            criterion,
                                            train_solver,
                                            spo_scale,
                                            mse_scale,
                                            gamma,
                                            X_val_tensor,
                                            Y_val_tensor,
                                            oracle_val_tensor,
                                            early_stopping_patience=2
                                            )

            # Inference
            
            # Scale test data using training scaler
            X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))

            X_test_tensor = torch.tensor(
                X_test_scaled,
                dtype=torch.float32
            )

            # Predict expected returns
            model.eval()
            with torch.no_grad():
                mu_hat_test = model(X_test_tensor)

            mu_hat_test = mu_hat_test.cpu().numpy()[0]

            test_mse = np.mean((mu_hat_test - Y_test)**2)

            # Compute portfolio weights from predicted returns with test solver
            test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix, test_index, num_scenarios=1000, random_seed=42)
            # test solver
            test_solver = CVaRSolver(
                loss_matrix=test_scenario_loss_matrix,
                N=N,
                S=S,
                alpha=0.95, # CVaR confidence level
                beta=beta
            )

            # Compute portfolio weights using test solver (train + val scenario matrix)
            w_hat = test_solver.solve(c=-mu_hat_test).copy()
            w_oracle = test_solver.solve(c=-Y_test).copy()

            # Realized returns
            realized_return = float(Y_test @ w_hat)
            oracle_return   = float(Y_test @ w_oracle)

            # True regret: how much return we lost by using predicted rather than true returns
            test_regret = oracle_return - realized_return

            results.append({

                # id
                "run_id": f"b{beta}_g{gamma}_t{test_index}",

                # experiment settings
                "beta": beta,
                "gamma": gamma,
                "test_index": test_index,

                # dimensions
                "n_train": len(X_train),
                "n_assets": N,
                "n_scenarios": S,

                # normalization factors
                "spo_scale": spo_scale,
                "mse_scale": mse_scale,

                # forecasting results
                "Y_hat_test": mu_hat_test,
                "Y_test": Y_test,
                "test_mse": test_mse,

                # portfolio results
                "weights": w_hat,
                "w_oracle": w_oracle,
                "realized_return": realized_return,
                "oracle_return": oracle_return,
                "test_regret": test_regret,

                # training history
                "history": history,
                "final_spo_loss": history["spo_loss"][-1],
                "final_mse_loss": history["mse_loss"][-1],
                "final_combined_loss": history["combined_loss"][-1],
                "early_stopping_epoch": history["early_stopping_epoch"],
                "best_epoch": history["best_epoch"],
                "best_val_regret": min(history["val_regret"])
            })

        
        # save results after every combination of beta and test_index
        with open("results_checkpoint_tmp.pkl", "wb") as f:
            pickle.dump(results, f)
        os.replace("results_checkpoint_tmp.pkl", "results_checkpoint.pkl")


test months:   0%|          | 0/1 [00:00<?, ?it/s]


Beta = 0.09; Test Index = 2

Preparing Data...


c:\Users\maxiw\master_thesis\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
Computing loss scales for normalization: 100%|██████████| 18/18 [00:28<00:00,  1.59s/it]


SPO+ scale: 0.995584 ; MSE  scale: 0.361648

Train Models for different loss combinations:

Gamma = 0.2 (weight of SPO+ loss)

Epoch 1: combined=0.850983 | SPO+ scaled=0.917981 | MSE scaled=0.834233 | val_regret=0.048978  ✓ regret improved
Epoch 2: combined=0.664095 | SPO+ scaled=0.793015 | MSE scaled=0.631865 | val_regret=0.050904


test months: 100%|██████████| 1/1 [02:31<00:00, 151.04s/it]

Epoch 3: combined=0.537138 | SPO+ scaled=0.708150 | MSE scaled=0.494384 | val_regret=0.050815
Early stopping at epoch 3. Best val regret: 0.048978


In [21]:
results

[{'run_id': 'b0.09_g0.2_t2',
  'beta': 0.09,
  'gamma': 0.2,
  'test_index': 2,
  'n_train': 278,
  'n_assets': 15,
  'n_scenarios': 1000,
  'spo_scale': np.float64(0.9955835024377814),
  'mse_scale': np.float64(0.36164815889464486),
  'Y_hat_test': array([ 0.31366384,  0.7074469 , -0.00689676,  0.26564264, -0.18454194,
          0.6263889 ,  0.17903368,  0.18635184,  0.17402007,  0.58733267,
          0.07700489, -0.21511969, -0.4789817 , -0.2263132 , -0.02736286],
        dtype=float32),
  'Y_test': array([ 0.08786545,  0.13485619,  0.08930863, -0.01811636,  0.05405818,
          0.08017352,  0.04797031, -0.00757436,  0.02985817,  0.07473122,
         -0.02918322,  0.06613175,  0.03264605, -0.05213261,  0.30942828]),
  'test_mse': np.float64(0.11054247852808302),
  'weights': array([ 0.03261048,  0.07719733, -0.        ,  0.00223997, -0.        ,
          0.2       ,  0.2       ,  0.14467528,  0.14327695,  0.2       ,
         -0.        , -0.        , -0.        , -0.        , -0. 

In [ ]:
# test performance has high variance in model initialization (seed) !!

In [11]:
with open("results_checkpoint.pkl", "rb") as f:
    results = pickle.load(f)

In [14]:
results[0]

{'run_id': 'b0.09_g0.2_t9',
 'beta': 0.09,
 'gamma': 0.2,
 'test_index': 9,
 'n_train': 271,
 'n_assets': 15,
 'n_scenarios': 1000,
 'spo_scale': np.float64(1.0303910480410445),
 'mse_scale': np.float64(0.3743181000737583),
 'Y_hat_test': array([ 0.15424395, -0.3051962 , -0.31012705,  0.15446168,  0.48376685,
        -0.115903  ,  0.12857725,  0.6790874 ,  0.1242224 , -0.45383793,
         0.13794142, -0.5394988 ,  0.14977168, -0.02002891,  0.26017904],
       dtype=float32),
 'Y_test': array([-0.01217478, -0.06609066,  0.03842863, -0.04302397, -0.06435433,
        -0.04382315, -0.09004948,  0.07108365, -0.13607595, -0.07983623,
        -0.06789244, -0.08260268, -0.0720169 ,  0.22525046, -0.12529449]),
 'test_mse': np.float64(0.11234353113222292),
 'weights': array([ 0.13596036, -0.        , -0.        ,  0.03650362,  0.2       ,
        -0.        ,  0.03839945,  0.2       ,  0.07429195, -0.        ,
        -0.        , -0.        ,  0.11484462, -0.        ,  0.2       ]),
 'w_oracle

## XX

12 months x 5 gamma levels x 4 risk aversion levels x 3 min training = 720min = 12h